# Olist E-Commerce: SQL Business Analysis

**Author:** Kunal &nbsp;|&nbsp; **Database:** MySQL 8 &nbsp;|&nbsp; **Dataset:** Brazilian E-Commerce Public Dataset by Olist (Sep 2016 – Oct 2018)

---

### Objective
Answer 9 real business questions about orders, delivery, payments, customers and sellers using **pure SQL**, from basic joins and aggregations to window functions, views, cohort analysis and query optimization.

### Skills Demonstrated
`JOINs` · `Aggregations` · `Date math` · `CTEs` · `Window functions (NTILE, DENSE_RANK, rolling frames)` · `Subqueries` · `Views` · `Cohort analysis` · `EXPLAIN / Indexing`

### Contents
| Tier | # | Topic |
|---|---|---|
| 1: Core Business Operations | Q1 | Delivery SLA breaches by state |
| | Q2 | Payment preferences and installments |
| | Q3 | Review score vs delivery time |
| 2: Advanced Analytics | Q4 | RFM customer segmentation |
| | Q5 | 7-day and 30-day rolling revenue |
| | Q6 | Top 3 sellers per category |
| 3: Views & Optimization | Q7 | Executive dashboard view |
| | Q8 | Monthly cohort retention |
| | Q9 | Query optimization with EXPLAIN |

## 1. Dataset Overview

11 CSV files loaded into a MySQL database named `olist` (about 1.4 million rows in total).

| Table | Rows | Grain (one row per…) |
|---|---:|---|
| `orders` | 99,441 | order |
| `order_items` | 112,650 | item inside an order |
| `order_payments` | 103,886 | payment record (an order can have several) |
| `order_reviews` | 99,224 | review (an order can have more than one) |
| `customers` | 99,441 | order-specific customer id (96,096 unique people) |
| `products` | 32,951 | product |
| `product_category_name_translation` | 71 | Portuguese → English category name |
| `sellers` | 3,095 | seller |
| `geolocation` | 1,000,163 | lat/lng sample (many rows per zip prefix) |
| `leads_qualified` | 8,000 | marketing qualified lead |
| `leads_closed` | 842 | closed (won) lead |

**Key relationships**

```
customers ──< orders ──< order_items >── products >── product_category_name_translation
                │              └────────< sellers 
                ├──< order_payments
                └──< order_reviews
customers / sellers ── zip_code_prefix ── geolocation
leads_qualified ── mql_id ── leads_closed ── seller_id ── sellers
```

## 2. Data Quality Notes

Findings from profiling the data **before** analysis. They drive several of the query decisions below.

- **Time range:** 2016-09-04 to 2018-10-17. 2016 and Sep–Oct 2018 contain very few orders, so time-series work uses **Jan 2017 – Aug 2018**.
- **Customer identity:** `customer_id` changes with every order. People are identified by **`customer_unique_id`** (96,096 unique vs 99,441 rows).
- **Repeat buying is rare:** about 96% of customers placed exactly one order.
- **Fan-out risks:** 2,961 orders have multiple payment rows and 547 have multiple reviews. `review_id` is not unique (789 repeated IDs). Aggregate to one row per order **before** joining.
- **Missing values:** 610 products have no category, and 2,965 orders have no delivery date (shipped, canceled, unavailable or invoiced orders). 775 orders have no items at all (mostly `unavailable` and `canceled`).
- **Translation gaps:** 73 product categories exist but only 71 have English names (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` are missing), so use `LEFT JOIN`.
- **Payment types:** `credit_card`, `boleto`, `voucher`, `debit_card` and 3 `not_defined` rows.

## 3. Setup: Connect Python to MySQL

Run the cell below once. Your password is typed into a hidden prompt, so it is never saved in the notebook.

> First time only: `pip install jupysql pymysql sqlalchemy pandas`

In [26]:
import os, getpass, time, statistics
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text

DB_USER, DB_HOST, DB_NAME = "root", "localhost", "olist"

password = os.environ.get("MYSQL_PWD") or getpass.getpass("MySQL password: ")
engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(password)}@{DB_HOST}:3306/{DB_NAME}")

%load_ext sql
%config SqlMagic.displaylimit = 25
%config SqlMagic.feedback = False
%sql engine


def time_query(sql, runs=3):
    """Run a query several times and report the median time (used in Q9)."""
    body = "\n".join(l for l in sql.splitlines() if l.strip() and not l.strip().startswith("--"))
    if not body:
        print("time_query needs a non-empty sql string.")
        return
    times = []
    with engine.connect() as conn:
        for _ in range(runs):
            start = time.perf_counter()
            conn.execute(text(sql)).fetchall()
            times.append(time.perf_counter() - start)
    print(f"Median: {statistics.median(times):.3f}s  |  runs: {[round(t, 3) for t in times]}")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


**Sanity check:** the row counts below should match the table in section 1.

In [27]:
%%sql
-- row counts: quick sanity check after loading the CSVs
SELECT 'orders' AS tbl, COUNT(*) AS row_count FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
UNION ALL SELECT 'customers', COUNT(*) FROM customers
UNION ALL SELECT 'geolocation', COUNT(*) FROM geolocation;

tbl,row_count
orders,99441
order_items,112650
customers,99441
geolocation,1000163


# Tier 1: Core Business Operations
*Joins, aggregations and date math*

---
## Q1. Delivery SLA Breaches

> What percentage of orders are delivered past their estimated delivery date? Group the results by customer state to see if specific geographic regions are experiencing logistical bottlenecks.

**Skills used:** `JOIN`, `CASE WHEN`, date comparison, `GROUP BY`

**Approach:** Counted delivered orders only and compared the delivered date with the estimated date at day level. Grouped by customer state and kept the order count next to the late rate, so tiny states don't mislead.

**1a. Overall late-delivery rate**

In [28]:
%%sql
-- Q1a: overall % of delivered orders that arrived after order_estimated_delivery_date
SELECT round( sum(case when date_format(order_delivered_customer_date, '%Y-%m-%d') > date_format(order_estimated_delivery_date, '%Y-%m-%d') 
     and order_status = 'delivered' then 1 else 0 end) *100 /
        sum(case when order_status = 'delivered' then 1 else 0 end),
2) as late_order_percentage
from orders

late_order_percentage
6.77


**1b. Late-delivery rate by customer state**

In [29]:
%%sql
-- Q1b: late-delivery rate by customer state

select c.customer_state,
       count(*) AS delivered_orders,
       sum(DATE(o.order_delivered_customer_date) > DATE(o.order_estimated_delivery_date)) as late_orders,
       round(100.0 * sum(DATE(o.order_delivered_customer_date) > DATE(o.order_estimated_delivery_date)) / count(*), 2) as late_order_percentage
from orders o
join customers c 
on c.customer_id = o.customer_id
where o.order_status = 'delivered'
  and o.order_delivered_customer_date IS NOT NULL
group by c.customer_state
order by late_order_percentage DESC ,delivered_orders DESC;

customer_state,delivered_orders,late_orders,late_order_percentage
AL,397,85,21.41
MA,717,125,17.43
SE,335,51,15.22
PI,476,66,13.87
CE,1279,176,13.76
RR,41,5,12.20
BA,3256,396,12.16
RJ,12350,1495,12.11
PA,946,106,11.21
ES,1995,214,10.73


**Insight:**

The worst states are AL (21.41%), MA (17.43%), SE (15.22%), PI (13.87%) and CE (13.76%). All five are in Brazil's Northeast, against a national rate of 6.77%.

RR (12.20%) has only 41 delivered orders, so treat it as noise. That's why the corrected query includes delivered_orders.

---
## Q2. Payment Preferences

> Which payment method (credit card, boleto, voucher) has the highest average order value? Do customers paying in multiple installments tend to spend more overall than those paying in a single lump sum?

**Skills used:** `GROUP BY`, subquery / CTE to get one row per order, `CASE` buckets

**Approach:** Payments are split across rows, so I summed `payment_value` per order first and then averaged by payment type. For installments I took the highest installment count per credit-card order and put it into buckets.

**2a. Average order value by payment method**

In [30]:
%%sql
-- Q2a: average order value by payment type

select t.payment_type,
       count(*) AS orders,
       round(avg(o.order_total), 2) as avg_order_value
from (select DISTINCT order_id, payment_type FROM order_payments) t
join (select order_id, SUM(payment_value) as order_total
      from order_payments group by order_id) o ON o.order_id = t.order_id
where t.payment_type <> 'not_defined'
group by t.payment_type
order by avg_order_value DESC;

payment_type,orders,avg_order_value
credit_card,76505,166.47
boleto,19784,145.03
debit_card,1528,142.73
voucher,3866,135.58


**2b. Installments vs total spend**

In [31]:
%%sql
-- Q2b: average order value by installment bucket (credit card only)

select case WHEN inst = 1 THEN '1 (lump sum)'
            WHEN inst BETWEEN 2 AND 3 THEN '2-3'
            WHEN inst BETWEEN 4 AND 6 THEN '4-6'
            ELSE '7+' END AS installment_bucket,
       count(*) as orders,
       round(avg(order_total), 2) as avg_order_value
from (select order_id,
             max(payment_installments) as inst,
             sum(payment_value) as order_total
      from order_payments
      where payment_type = 'credit_card' and payment_installments >= 1
      group by order_id) t
group by installment_bucket
order by min(inst);

installment_bucket,orders,avg_order_value
1 (lump sum),25333,96.17
2-3,22792,134.67
4-6,16205,181.79
7+,12173,336.02


**Insight:**

order value rises with the number of installments, and 7+ installment orders are about 3.5× a lump-sum order. Be careful with the wording. Big purchases are more likely to be split into installments, so this doesn't show that installments make people spend more.

---
## Q3. Review Sentiment Impact

> Calculate the average delivery time (in days) for 5-star reviews versus 1-star reviews. Does shipping delay strongly correlate with poor customer satisfaction?

**Skills used:** `JOIN`, `DATEDIFF`, aggregation, correlation

**Approach:** Joined orders to reviews, used `timestampdiff` for delivery days and compared 1-star with 5-star averages. MySQL has no `CORR()`, so I wrote Pearson's r by hand. Q3c then compares late vs on-time orders directly.

**3a. Average delivery days: 5-star vs 1-star**

In [32]:
%%sql
-- Q3a: average days from order_purchase_timestamp to order_delivered_customer_date, by review_score

select 
    round(avg(case when review_score = 5 then timestampdiff(DAY, order_purchase_timestamp, order_delivered_customer_date) else null end),2) as five_star_days,
    round(avg(case when review_score = 1 then timestampdiff(DAY, order_purchase_timestamp, order_delivered_customer_date) else null end),2) as one_star_days
from orders 
join order_reviews using(order_id);

five_star_days,one_star_days
10.22,20.85


**3b. Correlation between delivery days and review score** (MySQL has no `CORR()`, so compute Pearson manually in SQL, or pull the two columns into pandas)

In [33]:
%%sql
-- Q3b: Pearson correlation between delivery_days and review_score

with delivery_data as (
    select timestampdiff(day, o.order_purchase_timestamp, o.order_delivered_customer_date) as x,
           r.review_score as y
    from orders o
    join order_reviews r using (order_id)
    where o.order_delivered_customer_date is not null
)
select round(
    (sum(x * y) - count(*) * avg(x) * avg(y)) /
    (sqrt(sum(x * x) - count(*) * power(avg(x), 2)) *
     sqrt(sum(y * y) - count(*) * power(avg(y), 2))), 4) as correlation
from delivery_data;

correlation
-0.3337


**3c. Average review score: late vs on-time orders**

In [ ]:
%%sql
-- Q3c: average review score for late vs on-time delivered orders

select case when date(o.order_delivered_customer_date) > date(o.order_estimated_delivery_date)
            then 'late' else 'on_time' end as delivery_status,
       count(*) as reviews,
       round(avg(r.review_score), 2) as avg_review_score
from orders o
join order_reviews r using (order_id)
where o.order_status = 'delivered'
  and o.order_delivered_customer_date is not null
group by delivery_status;

**Insight:**

1-star orders take about twice as long to arrive as 5-star orders (20.85 vs 10.22 days).

The question asks about a strong correlation, and r = -0.33 is moderate. Longer delivery goes with lower scores, but delivery days explain only about 11% of the variation in scores (r² ≈ 0.11).

Lateness matters more than raw speed. Orders that arrived after the estimated date average 2.27 stars, against 4.29 for on-time orders (6,409 late vs 89,944 on-time reviews, see Q3c). Customers seem to react to a broken promise more than to slowness itself.

# Tier 2: Advanced Analytics
*CTEs, window functions and subqueries*

---
## Q4. RFM Customer Segmentation

> Use `NTILE()` window functions to build an RFM (Recency, Frequency, Monetary) model. Rank customers from 1 to 4 on how recently they bought, how often they buy, and how much they spend, classifying them into actionable buckets like "VIPs" and "Churn Risks."

**Skills used:** CTEs, `NTILE()`, `CASE`, `customer_unique_id`

**Approach:** Built R, F and M per `customer_unique_id`. Recency and monetary use `ntile(4)`. Frequency is capped at 4 with a `case` because 96% of customers ordered once and `ntile` would split those ties arbitrarily.

**4a. Build the R, F and M scores**

In [34]:
%%sql
-- Q4a: per customer_unique_id -> recency_days, frequency, monetary, then R/F/M scores (1-4)
with order_totals as (
    select order_id, sum(payment_value) as order_value
    from order_payments
    group by order_id
),
customer_base as (
    select c.customer_unique_id,
           max(o.order_purchase_timestamp) as last_order_date,
           count(distinct o.order_id)      as frequency,
           sum(ot.order_value)             as monetary
    from orders o
    join customers c on c.customer_id = o.customer_id
    join order_totals ot on ot.order_id = o.order_id
    where o.order_status not in ('canceled', 'unavailable')
    group by c.customer_unique_id
),
rfm as (
    select customer_unique_id,
           datediff((select max(last_order_date) from customer_base), last_order_date) as recency_days,
           frequency,
           round(monetary, 2) as monetary
    from customer_base
),
scored as (
    select *,
           ntile(4) over (order by recency_days desc) as r_score,
           case when frequency = 1 then 1
                when frequency = 2 then 2
                when frequency = 3 then 3
                else 4 end as f_score,
           ntile(4) over (order by monetary) as m_score
    from rfm
)
select * from scored
order by r_score desc, m_score desc
limit 10;

customer_unique_id,recency_days,frequency,monetary,r_score,f_score,m_score
064fb6f70338688d1372235d95d92ff7,116,1,1308.18,4,1,4
b5435c9ba16cf0cfcb89af488906838d,76,1,1147.55,4,1,4
8831da2a67b7007cd92faef855f9fb52,110,1,2055.84,4,1,4
fe203d7588a60b447f5c5f089b967097,108,1,1628.07,4,1,4
0fbad5f5683626b3c234ef751d2ff166,93,1,1437.64,4,1,4
ee1ab7f8a445e29acff592989e83a70a,70,1,1626.65,4,1,4
b0c3cd877e81291a25eb0be8b1919458,83,1,2045.47,4,1,4
a7acd7f8aefc1d48f618e3f15c5e327b,116,1,2036.04,4,1,4
a626420a40f09529d6dc67cc76e1b213,40,1,1150.80,4,1,4
5d1fde9eb27149c695624f8bb84cf6f7,25,1,1391.39,4,1,4


**4b. Segment the customers and size each segment**

In [35]:
%%sql
-- Q4b: classify into segments (VIPs, Loyal Repeat Buyers, New High-Value, New Customers, Churn Risks (High Value), Lost, Occasional) and count / revenue per segment
with order_totals as (
    select order_id, sum(payment_value) as order_value
    from order_payments
    group by order_id
),
customer_base as (
    select c.customer_unique_id,
           max(o.order_purchase_timestamp) as last_order_date,
           count(distinct o.order_id)      as frequency,
           sum(ot.order_value)             as monetary
    from orders o
    join customers c on c.customer_id = o.customer_id
    join order_totals ot on ot.order_id = o.order_id
    where o.order_status not in ('canceled', 'unavailable')
    group by c.customer_unique_id
),
rfm as (
    select customer_unique_id,
           datediff((select max(last_order_date) from customer_base), last_order_date) as recency_days,
           frequency,
           round(monetary, 2) as monetary
    from customer_base
),
scored as (
    select *,
           ntile(4) over (order by recency_days desc) as r_score,
           case when frequency = 1 then 1
                when frequency = 2 then 2
                when frequency = 3 then 3
                else 4 end as f_score,
           ntile(4) over (order by monetary) as m_score
    from rfm
)
select segment,
       count(*) as customers,
       round(100.0 * count(*) / sum(count(*)) over (), 2) as pct_of_customers,
       round(sum(monetary), 0) as total_revenue,
       round(100.0 * sum(monetary) / sum(sum(monetary)) over (), 2) as pct_of_revenue,
       round(avg(monetary), 2) as avg_spend
from (
    select *,
           case when f_score >= 2 and r_score >= 3 and m_score >= 3 then 'VIPs'
                when f_score >= 2 then 'Loyal Repeat Buyers'
                when r_score = 4 and m_score >= 3 then 'New High-Value'
                when r_score = 4 then 'New Customers'
                when r_score = 1 and m_score >= 3 then 'Churn Risks (High Value)'
                when r_score = 1 then 'Lost'
                else 'Occasional' end as segment
    from scored
) t
group by segment
order by total_revenue desc;

segment,customers,pct_of_customers,total_revenue,pct_of_revenue,avg_spend
Occasional,46017,48.44,7329244,46.57,159.27
New High-Value,11458,12.06,3053836,19.40,266.52
Churn Risks (High Value),10938,11.52,2962099,18.82,270.81
Lost,12208,12.85,777852,4.94,63.72
New Customers,11480,12.09,725573,4.61,63.20
VIPs,1405,1.48,482993,3.07,343.77
Loyal Repeat Buyers,1483,1.56,407541,2.59,274.81


**Insight:** Customers split into 7 segments. 'Occasional' is the largest (48.4% of customers, 46.6% of revenue). The one to act on is **Churn Risks (High Value)**: 11.5% of customers and 18.8% of revenue (R$ 2.96M), with an average spend of R$ 270.81, who have stopped buying. VIPs and Loyal Repeat Buyers together are only about 3% of customers, which matches the 96% one-time-buyer finding. Limitation: 'Occasional' is a broad catch-all, so the segment rules could be tightened.

---
## Q5. Moving Averages

> Calculate the 7-day and 30-day rolling average of daily revenue. This requires advanced window framing to smooth out weekend dips and identify true revenue trends over time.

**Skills used:** CTEs, `AVG() OVER (ROWS BETWEEN …)`, calendar table

**Approach:** Built a calendar with a recursive CTE so days with no sales count as zero, then used `avg() over (rows between 6 preceding and current row)` for the 7-day window and 29 preceding for the 30-day one. The output starts on the first day with a full 30-day window.

In [36]:
%%sql
-- Q5: daily revenue + 7-day and 30-day rolling average (Jan 2017 - Aug 2018)
with recursive calendar as (
    select date('2017-01-05') as dt
    union all
    select dt + interval 1 day from calendar where dt < '2018-08-22'
),
order_totals as (
    select order_id, sum(payment_value) as order_value
    from order_payments
    group by order_id
),
daily_revenue as (
    select date(o.order_purchase_timestamp) as dt,
           sum(ot.order_value) as revenue
    from orders o
    join order_totals ot on ot.order_id = o.order_id
    where o.order_status not in ('canceled', 'unavailable')
    group by date(o.order_purchase_timestamp)
),
daily as (
    select c.dt, coalesce(d.revenue, 0) as revenue
    from calendar c
    left join daily_revenue d on d.dt = c.dt
),
rolling as (
    select dt,
           revenue,
           avg(revenue) over (order by dt rows between 6 preceding and current row)  as rolling_7d_avg,
           avg(revenue) over (order by dt rows between 29 preceding and current row) as rolling_30d_avg
    from daily
)
select dt,
       round(revenue, 2)         as daily_revenue,
       round(rolling_7d_avg, 2)  as rolling_7d_avg,
       round(rolling_30d_avg, 2) as rolling_30d_avg
from rolling
where dt >= '2017-02-03'   -- first day that has a full 30-day window
order by dt;

dt,daily_revenue,rolling_7d_avg,rolling_30d_avg
2017-02-03,11399.86,9855.20,5610.77
2017-02-04,15717.21,11256.23,6111.10
2017-02-05,10153.24,11683.74,6416.60
2017-02-06,11086.04,11565.76,6737.87
2017-02-07,16454.83,12103.94,7259.37
2017-02-08,13960.93,12750.84,7698.92
2017-02-09,10844.17,12802.33,8008.02
2017-02-10,15052.88,13324.19,8413.73
2017-02-11,5233.15,11826.46,8504.75
2017-02-12,16603.91,12747.99,8965.52


**Insight:** The 30-day average climbs from R$ 5,611 (3 Feb 2017) to R$ 42,191 (22 Aug 2018), peaking at R$ 44,462 around 15 Dec 2017. The biggest single day was 24 Nov 2017 (R$ 178,450), far above its surrounding average, so the 30-day line is the one to read for trend and the daily numbers are mostly spikes.

---
## Q6. Seller Performance Ranks

> For each product category, rank the top 3 sellers by total revenue using `DENSE_RANK()`. Filter out any sellers who have an average customer review score below 4.0.

**Skills used:** CTEs, `DENSE_RANK() OVER (PARTITION BY …)`, filtering on aggregates

**Approach:** Ranked sellers inside each category with `dense_rank()` on revenue. Reviews are averaged per order first to avoid double counting, and sellers under a 4.0 average are filtered out before ranking.

In [37]:
%%sql
-- Q6: top 3 sellers per product category by revenue, only sellers with avg review score >= 4.0
with order_reviews_avg as (
    select order_id, avg(review_score) as review_score
    from order_reviews
    group by order_id
),
valid_items as (
    select oi.order_id, oi.seller_id, oi.price,
           coalesce(t.product_category_name_english, p.product_category_name) as category
    from order_items oi
    join orders o on o.order_id = oi.order_id
    join products p on p.product_id = oi.product_id
    left join product_category_name_translation t on t.product_category_name = p.product_category_name
    where o.order_status not in ('canceled', 'unavailable')
      and p.product_category_name is not null
),
seller_rating as (
    select s.seller_id, avg(r.review_score) as avg_review
    from (select distinct order_id, seller_id from valid_items) s
    join order_reviews_avg r on r.order_id = s.order_id
    group by s.seller_id
),
category_revenue as (
    select category, seller_id, sum(price) as revenue
    from valid_items
    group by category, seller_id
),
ranked as (
    select cr.category,
           cr.seller_id,
           round(cr.revenue, 2) as revenue,
           round(sr.avg_review, 2) as avg_review,
           dense_rank() over (partition by cr.category order by cr.revenue desc) as revenue_rank
    from category_revenue cr
    join seller_rating sr on sr.seller_id = cr.seller_id
    where sr.avg_review >= 4.0
)
select category, revenue_rank, seller_id, revenue, avg_review
from ranked
where revenue_rank <= 3
order by category, revenue_rank;

category,revenue_rank,seller_id,revenue,avg_review
agro_industry_and_commerce,1,e59aa562b9f8076dd550fcddf0e73491,31574.30,4.21
agro_industry_and_commerce,2,f08a5b9dd6767129688d001acafc21e5,7535.54,4.27
agro_industry_and_commerce,3,31ae0774c17fabd06ff707cc5bde005f,5169.89,4.00
air_conditioning,1,7901646fdd36a55f564ffaf2dbccaaf7,8900.08,4.50
air_conditioning,2,f3b80352b986ab4d1057a4b724be19d0,3320.00,4.17
air_conditioning,3,4391ea5c5990502176a1dcda44a2373d,2254.00,4.67
art,1,ee27a8f15b1dded4d213a468ba4eb391,6499.00,5.00
art,2,955fee9216a65b617aa5c0531780ce60,914.00,4.16
art,3,b561927807645834b59ef0d16ba55a24,676.20,4.34
arts_and_craftmanship,1,04ee0ec01589969663ba5967c0e0bdc0,289.49,5.00


**Insight:** Top sellers can dominate a category: in `agro_industry_and_commerce` the #1 seller made R$ 31,574 against R$ 7,535 for #2. Because the 4.0 review filter runs before the ranking, a high-revenue seller with poor reviews never appears in the list.

# Tier 3: Views and Optimization
*Views and optimization*

---
## Q7. The Reporting View

> Combine the Orders, Order Items, Products, and Customers tables into a single master `CREATE VIEW vw_executive_dashboard AS ...` that contains only the clean, non-null columns needed for an executive BI dashboard.

**Skills used:** `CREATE VIEW`, multi-table `JOIN`, `LEFT JOIN`, `COALESCE`, column selection

**Approach:** Joined orders, items, customers and products into one flat view of delivered items, using `left join` for the category tables and `coalesce` for an 'unknown' bucket. `delivery_days` and `is_late` are pre-computed so dashboards don't repeat that logic.

**7a. Create the view**

In [38]:
%%sql
-- Q7a: create or replace view vw_executive_dashboard (flat table of delivered order items for BI tools)
create or replace view vw_executive_dashboard as
select o.order_id,
       oi.order_item_id,
       date(o.order_purchase_timestamp)                         as order_date,
       year(o.order_purchase_timestamp)                         as order_year,
       date_format(o.order_purchase_timestamp, '%Y-%m')         as order_month,
       c.customer_unique_id,
       c.customer_state,
       c.customer_city,
       oi.product_id,
       coalesce(t.product_category_name_english, p.product_category_name, 'unknown') as product_category,
       oi.seller_id,
       oi.price,
       oi.freight_value,
       oi.price + oi.freight_value                              as item_total,
       datediff(o.order_delivered_customer_date, o.order_purchase_timestamp) as delivery_days,
       case when date(o.order_delivered_customer_date) > date(o.order_estimated_delivery_date)
            then 1 else 0 end                                   as is_late
from orders o
join order_items oi on oi.order_id = o.order_id
join customers c on c.customer_id = o.customer_id
left join products p on p.product_id = oi.product_id
left join product_category_name_translation t on t.product_category_name = p.product_category_name
where o.order_status = 'delivered'
  and o.order_delivered_customer_date is not null;

++
||
++
++

**7b. Preview the view**

In [39]:
%%sql

select * from vw_executive_dashboard limit 10;

order_id,order_item_id,order_date,order_year,order_month,customer_unique_id,customer_state,customer_city,product_id,product_category,seller_id,price,freight_value,item_total,delivery_days,is_late
00010242fe8c5a6d1ba2dd792cb16214,1,2017-09-13,2017,2017-09,871766c5855e863f6eccc05f988b23cb,RJ,campos dos goytacazes,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,48436dade18ac8b2bce089ec2a041202,58.90,13.29,72.19,7,0
00018f77f2f0320c557190d7a144bdd3,1,2017-04-26,2017,2017-04,eb28e67c4c0b83846050ddfb8a35d051,SP,santa fe do sul,e5f2d52b802189ee658865ca93d83a8f,pet_shop,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93,259.83,16,0
000229ec398224ef6ca0657da4fc703e,1,2018-01-14,2018,2018-01,3818d81c6709e39d06b2738a8d3a2474,MG,para de minas,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,5b51032eddd242adc84c38acab88f23d,199.00,17.87,216.87,8,0
00024acbcdf0a6daa1e931b038114c75,1,2018-08-08,2018,2018-08,af861d436cfc08b2c2ddefd0ba074622,SP,atibaia,7634da152a4610f1595efa32f14722fc,perfumery,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79,25.78,6,0
00042b26cf59d7ce69dfabb4e55b4fd9,1,2017-02-04,2017,2017-02,64b576fb70d441e8f1b2d7d446e483c5,SP,varzea paulista,ac6c3623068f30de03045865e4e10089,garden_tools,df560393f3a51e74553ab94004ba5c87,199.90,18.14,218.04,25,0
00048cc3ae777c65dbb7d2a0634bc1ea,1,2017-05-15,2017,2017-05,85c835d128beae5b4ce8602c491bf385,MG,uberaba,ef92defde845ab8450f9d70c526ef70f,housewares,6426d21aca402a131fc0a5d0960a3c90,21.90,12.69,34.59,7,0
00054e8431b9d7675808bcb819fb4a32,1,2017-12-10,2017,2017-12,635d9ac1680f03288e72ada3a1035803,SP,guararapes,8d4f2bb7e93e6710a28f34fa83ee7d28,telephony,7040e82f899a04d1b434b795a43b4617,19.90,11.85,31.75,8,0
000576fe39319847cbb9d288c5617fa6,1,2018-07-04,2018,2018-07,fda4476abb6307ab3c415b7e6d026526,SP,praia grande,557d850972a7d6f792fd18ae1400d9b6,garden_tools,5996cddab893a4652a15592fb58ab8db,810.00,70.75,880.75,5,0
0005a1a1728c9d785b8e2b08b904576c,1,2018-03-19,2018,2018-03,639d23421f5517f69d0c3d6e6564cf0e,SP,santos,310ae3c140ff94b03219ad0adc3c778f,health_beauty,a416b6a846a11724393025641d4edd5e,145.95,11.65,157.60,10,0
0005f50442cb953dcd1d21e1fb923495,1,2018-07-02,2018,2018-07,0782c41380992a5a533489063df0eef6,SP,jandira,4535b0e1091c278dfd193e5a1d63b39f,books_technical,ba143b05f0110f0dc71ad71b4466ce92,53.99,11.40,65.39,2,0


**Insight:** `vw_executive_dashboard` gives a BI tool one flat table (one row per delivered order item) with `delivery_days` and `is_late` already calculated, so dashboards don't repeat the joins. Orders without a delivered date are excluded on purpose.

---
## Q8. Cohort Retention Analysis

> Write a complex query using self-joins or CTEs to calculate monthly retention. If 1,000 users made their first purchase in January, what percentage of those specific users returned to make another purchase in February, March, and April?

**Skills used:** CTEs, self-join, `TIMESTAMPDIFF`, conditional aggregation, `customer_unique_id`

**Approach:** Found each customer's first purchase month (their cohort), then counted how many came back 1, 2 and 3 months later with `timestampdiff`. The percentages are shares of the cohort, so 0.40 means 0.40%.

In [40]:
%%sql
-- Q8: monthly cohort retention (cohort month, cohort size, % returning in month +1, +2, +3)
with customer_months as (
    select c.customer_unique_id,
           date_format(o.order_purchase_timestamp, '%Y-%m-01') as order_month
    from orders o
    join customers c on c.customer_id = o.customer_id
    where o.order_status not in ('canceled', 'unavailable')
    group by c.customer_unique_id, date_format(o.order_purchase_timestamp, '%Y-%m-01')
),
cohorts as (
    select customer_unique_id, min(order_month) as cohort_month
    from customer_months
    group by customer_unique_id
),
activity as (
    select co.cohort_month,
           co.customer_unique_id,
           timestampdiff(month, co.cohort_month, cm.order_month) as month_number
    from cohorts co
    join customer_months cm on cm.customer_unique_id = co.customer_unique_id
)
select date_format(cohort_month, '%Y-%m') as cohort,
       count(distinct customer_unique_id) as cohort_size,
       round(100.0 * count(distinct case when month_number = 1 then customer_unique_id end) / count(distinct customer_unique_id), 2) as month_1_pct,
       round(100.0 * count(distinct case when month_number = 2 then customer_unique_id end) / count(distinct customer_unique_id), 2) as month_2_pct,
       round(100.0 * count(distinct case when month_number = 3 then customer_unique_id end) / count(distinct customer_unique_id), 2) as month_3_pct
from activity
where cohort_month between '2017-01-01' and '2018-05-01'
group by cohort_month
order by cohort_month;

cohort,cohort_size,month_1_pct,month_2_pct,month_3_pct
2017-01,752,0.40,0.27,0.13
2017-02,1690,0.24,0.30,0.12
2017-03,2571,0.51,0.35,0.39
2017-04,2325,0.60,0.22,0.17
2017-05,3541,0.48,0.48,0.40
2017-06,3102,0.45,0.35,0.39
2017-07,3822,0.52,0.34,0.24
2017-08,4130,0.68,0.34,0.27
2017-09,4075,0.69,0.54,0.29
2017-10,4392,0.71,0.25,0.09


**Insight:** Retention is very low. Month-1 return rate sits between 0.22% and 0.71% across all 17 cohorts, which fits the 96% one-time-buyer finding. Cohorts also grew from 752 customers (Jan 2017) to 7,190 (Nov 2017), so growth depended on acquiring new customers rather than bringing buyers back.

---
## Q9. Query Optimization

> Take your heaviest query (likely the Cohort Analysis). Run `EXPLAIN` on it. Identify which `JOIN` or `WHERE` clause is scanning the most rows, apply a `CREATE INDEX` on that foreign key, and document the before-and-after execution times.

**Skills used:** `EXPLAIN` / `EXPLAIN ANALYZE`, `CREATE INDEX`, performance benchmarking

**Approach:** Ran `explain analyze` on a join-heavy seller query, found which step scanned the most rows, indexed that join key and compared the plans and timings.

**9-0. Reset (so Q9 can be re-run from the top)**

This drops the two demo indexes if they exist. It keeps the 'before' timing honest and avoids the *Duplicate key name* error if `create index` is run twice.

In [ ]:
from sqlalchemy import text

# drop the demo indexes if they exist, so Q9 can be re-run from the top
for tbl, idx in [("order_reviews", "idx_reviews_order"), ("order_items", "idx_items_seller")]:
    with engine.begin() as conn:
        found = conn.execute(text(
            "select count(*) from information_schema.statistics "
            "where table_schema = database() and table_name = :t and index_name = :i"
        ), {"t": tbl, "i": idx}).scalar()
        if found:
            conn.execute(text(f"drop index {idx} on {tbl}"))
            print("dropped", idx)
        else:
            print("not present:", idx)

**9a. The heaviest query:** timed before indexing (median of 3 runs)

In [42]:
time_query("""
select oi.seller_id,
       count(*) as items_sold,
       round(sum(oi.price), 2) as revenue,
       round(avg(r.review_score), 2) as avg_review
from order_items oi
join order_reviews r on r.order_id = oi.order_id
where oi.seller_id = '4a3ca9315b744ce9f8e9374361493884'
group by oi.seller_id;
""")

Median: 1.214s  |  runs: [1.206, 1.214, 1.221]


**9b. EXPLAIN before indexing:** which step scans the most rows?

In [43]:
%%sql
-- Q9b: explain analyze on the seller-performance query, before indexing
explain analyze
select oi.seller_id,
       count(*) as items_sold,
       round(sum(oi.price), 2) as revenue,
       round(avg(r.review_score), 2) as avg_review
from order_items oi
join order_reviews r on r.order_id = oi.order_id
where oi.seller_id = '4a3ca9315b744ce9f8e9374361493884'
group by oi.seller_id;

EXPLAIN
"-> Group aggregate: avg(r.review_score), sum(oi.price), count(0) (cost=47184 rows=334) (actual time=1284..1284 rows=1 loops=1) -> Nested loop inner join (cost=46065 rows=11196) (actual time=0.113..1281 rows=1984 loops=1) -> Table scan on r (cost=10128 rows=98962) (actual time=0.0304..140 rows=99224 loops=1) -> Filter: (oi.seller_id = '4a3ca9315b744ce9f8e9374361493884') (cost=0.25 rows=0.113) (actual time=0.0111..0.0111 rows=0.02 loops=99224) -> Index lookup on oi using PRIMARY (order_id=r.order_id) (cost=0.25 rows=1.13) (actual time=0.00909..0.0105 rows=1.13 loops=99224)"


**9c. Create the index**

In [44]:
%%sql
-- Q9c: index the join key order_reviews(order_id)
create index idx_reviews_order on order_reviews(order_id);


++
||
++
++

**9d. EXPLAIN after indexing**

In [45]:
%%sql
-- Q9d: explain analyze again, after indexing
explain analyze
select oi.seller_id,
       count(*) as items_sold,
       round(sum(oi.price), 2) as revenue,
       round(avg(r.review_score), 2) as avg_review
from order_items oi
join order_reviews r on r.order_id = oi.order_id
where oi.seller_id = '4a3ca9315b744ce9f8e9374361493884'
group by oi.seller_id;

EXPLAIN
"-> Group aggregate: avg(r.review_score), sum(oi.price), count(0) (cost=16519 rows=334) (actual time=102..102 rows=1 loops=1) -> Nested loop inner join (cost=15397 rows=11220) (actual time=0.149..101 rows=1984 loops=1) -> Filter: (oi.seller_id = '4a3ca9315b744ce9f8e9374361493884') (cost=11470 rows=11173) (actual time=0.105..38.7 rows=1987 loops=1) -> Table scan on oi (cost=11470 rows=111725) (actual time=0.0952..30.2 rows=112650 loops=1) -> Index lookup on r using idx_reviews_order (order_id=oi.order_id) (cost=0.251 rows=1) (actual time=0.0298..0.0309 rows=0.998 loops=1987)"


**9e. Timed again after indexing** (median of 3 runs)

In [ ]:
time_query("""
select oi.seller_id,
       count(*) as items_sold,
       round(sum(oi.price), 2) as revenue,
       round(avg(r.review_score), 2) as avg_review
from order_items oi
join order_reviews r on r.order_id = oi.order_id
where oi.seller_id = '4a3ca9315b744ce9f8e9374361493884'
group by oi.seller_id;
""")

**Before vs after**

| Query | Bottleneck found in EXPLAIN | Index added | Before (s) | After (s) | Speed-up |
|---|---|---|---:|---:|---:|
| Seller performance (`order_items` joined to `order_reviews`) | Full scan of `order_reviews` (99,224 rows) driving a nested loop | `idx_reviews_order` on `order_reviews(order_id)` | 1.284 | 0.102 | about 12.6x |

_Times are from `explain analyze`. The `time_query` median before indexing was 1.214 s; re-run 9e for the matching after-index median._

> Indexes do not always help. If a join returns many rows per key, a full scan with a hash join can win. Note it in your write-up if you see that.

**Insight:** The plan showed a full scan of all 99,224 `order_reviews` rows driving a nested loop (about 1,284 ms in `explain analyze`). After indexing `order_reviews(order_id)` the join became an index lookup and the query ran in about 102 ms, roughly 12.6x faster. The plan still scans `order_items` to filter by seller, so `order_items(seller_id)` is the next candidate. Indexes speed up reads but add write cost, so I would only add them on join keys that are used often.

---
# Summary of Findings

| # | Question | Key finding |
|---|---|---|
| Q1 | Delivery SLA | 6.77% of delivered orders arrived late. AL is worst at 21.41% and all five worst states are in the Northeast. |
| Q2 | Payment preferences | Credit card has the highest average order value (R$ 166.47). 7+ installments is about 3.5x a lump sum, which is correlation, not cause. |
| Q3 | Reviews vs delivery | 1-star orders took 20.85 days vs 10.22 for 5-star (r = -0.33). Late orders average 2.27 stars vs 4.29 on time. |
| Q4 | RFM segments | High-value churn risks are 11.5% of customers but 18.8% of revenue. |
| Q5 | Rolling revenue | The 30-day average shows the real trend; daily revenue is mostly spikes. |
| Q6 | Top sellers | A single seller can dominate a category; the 4.0 review filter removes poorly rated sellers before ranking. |
| Q7 | Executive view | One flat view of delivered items with delivery_days and is_late ready for BI. |
| Q8 | Cohort retention | Month-1 retention is only 0.22% to 0.71% across cohorts. |
| Q9 | Optimization | Index on order_reviews(order_id): about 1,284 ms to 102 ms (12.6x). |

## Business Recommendations
1. **Delivery promises (Q1, Q3):** late orders average 2.27 stars vs 4.29 on time, so it is better to promise a safer date than to miss one. The Northeast lanes (AL, MA, SE, PI, CE) need carrier attention first.
2. **Retention (Q4, Q8):** under 1% of customers return the next month. Run a win-back offer for the high-value churn-risk segment (18.8% of revenue) and test a second-purchase incentive in the first 30 days.
3. **Planning and sellers (Q5, Q6):** plan on the 30-day average because daily revenue is spiky, and protect the top sellers per category, since in some categories one seller carries most of the revenue.

## Limitations
- Data covers 2016–2018 only, and 2016 / late 2018 are sparse.
- Most customers bought once, so retention and RFM frequency are based on small repeat-buyer groups.
- The Q4 segment rules are my own choices, and 'Occasional' (48% of customers) is a broad catch-all.
- Q8 percentages are small because most customers buy once, and cohort sizes differ a lot.
- Revenue is based on item price and payment value as recorded. Refunds are not available.

---
*Author: Kunal · [GitHub](https://github.com/Kunal-singh-99) · [LinkedIn](https://www.linkedin.com/in/kunal-jadon-a9796735b/?isSelfProfile=true)*